# Experiment 7c — DL-track submission (LGBM-MAE + MLP-MAE blend)

Production refit + leaderboard submission for the blend chosen in `kp-experiment-7b-blend.ipynb`:

- **Weights** (frozen from 7b §7, optimal pooled): `LGBM = 0.285`, `MLP = 0.715`
- **Expected CV-WAPE**: 14.47 (per 7b §4 — pooled-weight 2-way blend)
- **Seq2seq excluded** (3-way over 2-way = +0.006 WAPE, below the 0.05 noise floor)

Both base models are refit on the **full** train (no CV holdout) and run on `validation_input.csv`. MLP uses 5 seeds per the README's production convention (the diagnostic in 7a used 3); LGBM also uses 5 seeds for symmetry and free variance reduction. Feature engineering is the verbatim exp-5.1 / 7a pipeline applied to the **combined** train+val frame so lag/lead features at the val boundary have proper context.

Output: `validation_predictions_7c.csv` in the standard `(series_id, timestamp, prediction)` submission schema, plus a verbose `val_components_7c.csv` with the per-model components for future diagnostic work.

---
## §0. Configuration

Paths and feature config mirror exp-5.1 / exp-7a-tabular exactly. MLP hyperparameters match 7a (the model that produced the OOFs the blend weights were fit on). Blend weights are *frozen constants* — they came from 7b's OOF optimization and changing them here would invalidate the diagnostic chain.

In [1]:
from pathlib import Path

# ── Paths ─────────────────────────────────────────────────────────────────────
# Kaggle paths (default):
DATA_DIR     = Path("/kaggle/input/datasets/xitij27/dlam-bonus-project-dataset/data")
OUTPUT_DIR   = Path("/kaggle/working")
# Local paths:
# DATA_DIR     = Path("data")
# OUTPUT_DIR   = Path("experiment-7")

TRAIN_PATH     = DATA_DIR / "train.csv"
VAL_INPUT_PATH = DATA_DIR / "validation_input.csv"
VAL_INDEX_PATH = DATA_DIR / "forecast_index_validation.csv"

# ── Reproducibility ──────────────────────────────────────────────────────────
SEED         = 42
SEEDS        = [42, 101, 202, 303, 404]   # 5 seeds — production convention from exp-5.1 / Phase 7 README
EVAL_HORIZON = 336                        # for reference; not used at submission time

# ── Feature config — IDENTICAL to exp-5.1 / 7a-tabular ───────────────────────
DOMAIN_FEAT_COLS = [
    "workload_intensity", "demand_forecast", "upstream_quality_forecast",
    "promotion_intensity", "shock_risk", "maintenance_known",
    "unit_reliability_forecast", "queue_pressure_forecast",
    "network_pressure_forecast", "event_load_forecast",
    "service_irregularity_risk_forecast",
]
TIME_COLS = ["hour_sin", "hour_cos", "dow_sin", "dow_cos",
             "doy_sin", "doy_cos", "is_weekend"]
TOP_DOM   = ["queue_pressure_forecast", "network_pressure_forecast",
             "workload_intensity", "shock_risk", "event_load_forecast"]
TOP3_ROLL = TOP_DOM[:3]
LAGS_DOM  = [1, 24]
LEADS_DOM = [1, 24]
ROLL_WINS = [6, 24]

# ── LGBM-MAE config (exp-5.1 BASELINE_LGBM) ──────────────────────────────────
BASELINE_LGBM = dict(
    n_estimators=600, learning_rate=0.03, num_leaves=63,
    subsample=0.8, colsample_bytree=0.8, min_child_samples=50,
    reg_lambda=1.0, verbose=-1,
)

# ── MLP config — IDENTICAL to 7a-tabular ─────────────────────────────────────
MLP_HIDDEN       = [256, 256, 128]
MLP_DROPOUT      = 0.2
MLP_EMBED_DIM    = 16
MLP_BATCH        = 1024
MLP_LR           = 1e-3
MLP_WEIGHT_DECAY = 1e-4
MLP_EPOCHS       = 30
MLP_EARLY_STOP   = 5    # patience on inner 10% val

# ── Frozen blend weights from experiment-7b §7 ───────────────────────────────
W_LGBM = 0.285
W_MLP  = 0.715
assert abs(W_LGBM + W_MLP - 1.0) < 1e-9, "blend weights must sum to 1"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"DATA_DIR    = {DATA_DIR}")
print(f"OUTPUT_DIR  = {OUTPUT_DIR}")
print(f"Seeds       : {SEEDS}")
print(f"Blend       : {W_LGBM:.3f} · LGBM-MAE  +  {W_MLP:.3f} · MLP-MAE")

DATA_DIR    = /kaggle/input/datasets/xitij27/dlam-bonus-project-dataset/data
OUTPUT_DIR  = /kaggle/working
Seeds       : [42, 101, 202, 303, 404]
Blend       : 0.285 · LGBM-MAE  +  0.715 · MLP-MAE


---
## §1. Setup & data loading

Verbatim copy of exp-7a-tabular §1 — same imports, same sid mapping, same WAPE helper, same device check.

In [2]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import lightgbm as lgb

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from sklearn.preprocessing import RobustScaler

np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"  GPU: {torch.cuda.get_device_name(0)}")

def wape(y_true, y_pred, eps=1e-8):
    return np.sum(np.abs(np.asarray(y_true) - np.asarray(y_pred))) / \
           (np.sum(np.abs(y_true)) + eps) * 100

train_df  = pd.read_csv(TRAIN_PATH,     parse_dates=["timestamp"])
val_input = pd.read_csv(VAL_INPUT_PATH, parse_dates=["timestamp"])
val_index = pd.read_csv(VAL_INDEX_PATH, parse_dates=["timestamp"])

sid_to_code = {sid: i for i, sid in enumerate(sorted(train_df["series_id"].unique()))}
N_SERIES = len(sid_to_code)

print(f"train_df  : {train_df.shape}")
print(f"val_input : {val_input.shape}")
print(f"val_index : {val_index.shape}")
print(f"# series  : {N_SERIES}")

PyTorch device: cuda
  GPU: Tesla T4
train_df  : (414720, 25)
val_input : (32256, 24)
val_index : (32256, 2)
# series  : 96


---
## §2. Feature engineering — exp-5.1 / 7a-tabular pipeline

Same `impute_domain_features` / `add_time_features` / `add_temporal_features`. Difference from CV notebooks: features are computed on the **combined** train+val frame so that the lag/lead/rolling features at the start of the val window have proper context from the tail of train, instead of being NaN-filled.

In [3]:
def impute_domain_features(df, cols):
    df = df.sort_values(["series_id", "timestamp"]).copy()
    df[cols] = df.groupby("series_id", group_keys=False)[cols].transform(
        lambda x: x.ffill().bfill()
    )
    return df

def add_time_features(df, time_col="timestamp"):
    df = df.copy()
    t = pd.to_datetime(df[time_col])
    df["hour_sin"]   = np.sin(2 * np.pi * t.dt.hour / 24)
    df["hour_cos"]   = np.cos(2 * np.pi * t.dt.hour / 24)
    df["dow_sin"]    = np.sin(2 * np.pi * t.dt.dayofweek / 7)
    df["dow_cos"]    = np.cos(2 * np.pi * t.dt.dayofweek / 7)
    df["doy_sin"]    = np.sin(2 * np.pi * t.dt.dayofyear / 365)
    df["doy_cos"]    = np.cos(2 * np.pi * t.dt.dayofyear / 365)
    df["is_weekend"] = (t.dt.dayofweek >= 5).astype(float)
    return df

def add_temporal_features(df, lag_cols, roll_cols, lags, leads, windows):
    df = df.copy().sort_values(["series_id", "timestamp"])
    new = []
    for col in lag_cols:
        g = df.groupby("series_id", sort=False)[col]
        for L in lags:
            c = f"{col}_lag{L}";  df[c] = g.shift(L);  new.append(c)
        for L in leads:
            c = f"{col}_lead{L}"; df[c] = g.shift(-L); new.append(c)
    for col in roll_cols:
        g = df.groupby("series_id", sort=False)[col]
        for W in windows:
            c_m = f"{col}_roll{W}m"
            df[c_m] = g.shift(1).transform(lambda x: x.rolling(W, min_periods=1).mean())
            new.append(c_m)
            c_s = f"{col}_roll{W}s"
            df[c_s] = g.shift(1).transform(lambda x: x.rolling(W, min_periods=1).std().fillna(0))
            new.append(c_s)
    return df, new

# Build combined frame (val rows carry target = NaN as the stub)
val_stub = val_input.copy()
val_stub["target"] = np.nan
combined = pd.concat([train_df, val_stub], ignore_index=True)

combined = add_time_features(impute_domain_features(combined, DOMAIN_FEAT_COLS))
combined["sid_code"] = combined["series_id"].map(sid_to_code).astype("int32")

# series_mean — computed from train only (leakage-free; val rows inherit by merge)
sm_full = train_df.groupby("series_id")["target"].mean().rename("series_mean").reset_index()
combined = combined.merge(sm_full, on="series_id")

combined, TEMPORAL_FEATS = add_temporal_features(
    combined, lag_cols=TOP_DOM, roll_cols=TOP3_ROLL,
    lags=LAGS_DOM, leads=LEADS_DOM, windows=ROLL_WINS,
)

NUMERIC_FEATS  = DOMAIN_FEAT_COLS + TIME_COLS + ["series_mean"] + TEMPORAL_FEATS
FEATS_WITH_SID = NUMERIC_FEATS + ["sid_code"]

train_part = combined[combined["target"].notna()].reset_index(drop=True)
val_part   = combined[combined["target"].isna()].reset_index(drop=True)

print(f"Features          : {len(FEATS_WITH_SID)}  (numeric {len(NUMERIC_FEATS)} + sid)")
print(f"Train rows        : {len(train_part):,}")
print(f"Val rows          : {len(val_part):,}")
assert len(val_part) == len(val_input), f"Val row count mismatch: {len(val_part)} vs {len(val_input)}"
assert len(train_part) == len(train_df), f"Train row count mismatch: {len(train_part)} vs {len(train_df)}"

Features          : 52  (numeric 51 + sid)
Train rows        : 414,720
Val rows          : 32,256


---
## §3. LGBM-MAE refit (5 seeds) → val predictions

Same `BASELINE_LGBM` config used in exp-5.1 and 7a, MAE objective, sid_code as the single categorical feature. 5 seeds averaged.

In [4]:
import time

print(f"Fitting LGBM-MAE × {len(SEEDS)} seeds on full train ({len(train_part):,} rows)...")
t0 = time.time()
lgbm_seed_preds = []
for s in SEEDS:
    model = lgb.LGBMRegressor(**BASELINE_LGBM, objective="regression_l1", random_state=s)
    model.fit(
        train_part[FEATS_WITH_SID],
        train_part["target"].to_numpy(float),
        categorical_feature=["sid_code"],
    )
    p = model.predict(val_part[FEATS_WITH_SID])
    lgbm_seed_preds.append(p)
    print(f"  seed {s:>4}: mean pred = {p.mean():.3f}, std = {p.std():.3f}")

pred_lgbm = np.mean(np.stack(lgbm_seed_preds, axis=0), axis=0)
pred_lgbm = np.clip(pred_lgbm, 0.0, None)
print(f"\nLGBM ensemble done in {time.time() - t0:.1f}s.")
print(f"  pred_lgbm: shape {pred_lgbm.shape}, range [{pred_lgbm.min():.3f}, {pred_lgbm.max():.3f}], mean {pred_lgbm.mean():.3f}")

Fitting LGBM-MAE × 5 seeds on full train (414,720 rows)...
  seed   42: mean pred = 10.499, std = 5.149
  seed  101: mean pred = 10.491, std = 5.168
  seed  202: mean pred = 10.495, std = 5.158
  seed  303: mean pred = 10.502, std = 5.158
  seed  404: mean pred = 10.491, std = 5.161

LGBM ensemble done in 207.7s.
  pred_lgbm: shape (32256,), range [1.279, 35.497], mean 10.496


---
## §4. MLP-MAE refit (5 seeds) → val predictions

Architecture and training procedure verbatim from 7a-tabular §5 — including the inner 10% holdout for early stopping. Each seed reuses the same `prepare_fold_tensors` helper but with `train_part` as the train set and `val_part` as the inference target (target column is NaN for val, never read except via the unused `te_y_raw` field).

Predictions are un-normalized per series before averaging. Same `np.clip(..., 0, None)` as in CV.

In [5]:
class SeriesMLP(nn.Module):
    def __init__(self, n_numeric, n_series, embed_dim, hidden_dims, dropout):
        super().__init__()
        self.series_emb = nn.Embedding(n_series, embed_dim)
        in_dim = n_numeric + embed_dim
        layers = []
        for h in hidden_dims:
            layers += [nn.Linear(in_dim, h), nn.BatchNorm1d(h),
                       nn.ReLU(inplace=True), nn.Dropout(dropout)]
            in_dim = h
        layers.append(nn.Linear(in_dim, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x_num, sid):
        e = self.series_emb(sid)
        x = torch.cat([x_num, e], dim=1)
        return self.net(x).squeeze(-1)


def prepare_tensors(tr_df, te_df, numeric_feats, sid_col="sid_code", target_col="target"):
    "Fit RobustScaler on tr_df, transform tr+te, compute per-series μ/σ from tr_df only."
    feat_scaler = RobustScaler().fit(tr_df[numeric_feats].astype(float))
    tr_X = np.nan_to_num(feat_scaler.transform(tr_df[numeric_feats].astype(float)), nan=0.0)
    te_X = np.nan_to_num(feat_scaler.transform(te_df[numeric_feats].astype(float)), nan=0.0)

    stats = tr_df.groupby(sid_col)[target_col].agg(["mean", "std"]).reset_index()
    stats["std"] = stats["std"].fillna(1.0).clip(lower=1e-3)
    mu = stats.set_index(sid_col)["mean"]
    sd = stats.set_index(sid_col)["std"]
    tr_mu = tr_df[sid_col].map(mu).to_numpy()
    tr_sd = tr_df[sid_col].map(sd).to_numpy()
    te_mu = te_df[sid_col].map(mu).to_numpy()
    te_sd = te_df[sid_col].map(sd).to_numpy()

    tr_y_norm = (tr_df[target_col].to_numpy(float) - tr_mu) / tr_sd

    return dict(
        tr_X=torch.tensor(tr_X, dtype=torch.float32),
        te_X=torch.tensor(te_X, dtype=torch.float32),
        tr_sid=torch.tensor(tr_df[sid_col].to_numpy(np.int64)),
        te_sid=torch.tensor(te_df[sid_col].to_numpy(np.int64)),
        tr_y_norm=torch.tensor(tr_y_norm, dtype=torch.float32),
        te_mu=te_mu, te_sd=te_sd,
    )


def train_mlp_one_seed(tensors, n_numeric, seed):
    torch.manual_seed(seed)
    np.random.seed(seed)

    model = SeriesMLP(n_numeric=n_numeric, n_series=N_SERIES,
                      embed_dim=MLP_EMBED_DIM, hidden_dims=MLP_HIDDEN,
                      dropout=MLP_DROPOUT).to(DEVICE)
    opt   = torch.optim.AdamW(model.parameters(), lr=MLP_LR, weight_decay=MLP_WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=MLP_EPOCHS)
    loss_fn = nn.L1Loss()

    # Inner 10% holdout for early stopping — same protocol as 7a CV.
    n_tr = tensors["tr_X"].shape[0]
    perm = torch.randperm(n_tr, generator=torch.Generator().manual_seed(seed))
    cut  = int(0.9 * n_tr)
    tr_idx, val_idx = perm[:cut], perm[cut:]

    tr_loader = DataLoader(
        TensorDataset(tensors["tr_X"][tr_idx], tensors["tr_sid"][tr_idx], tensors["tr_y_norm"][tr_idx]),
        batch_size=MLP_BATCH, shuffle=True, drop_last=False,
    )
    val_X     = tensors["tr_X"][val_idx].to(DEVICE)
    val_sid   = tensors["tr_sid"][val_idx].to(DEVICE)
    val_y     = tensors["tr_y_norm"][val_idx].numpy()

    best_val, patience, best_state, used_epochs = float("inf"), 0, None, 0
    for epoch in range(MLP_EPOCHS):
        model.train()
        for x, sid, y in tr_loader:
            x, sid, y = x.to(DEVICE), sid.to(DEVICE), y.to(DEVICE)
            opt.zero_grad()
            loss = loss_fn(model(x, sid), y)
            loss.backward()
            opt.step()
        sched.step()

        model.eval()
        with torch.no_grad():
            val_pred = model(val_X, val_sid).cpu().numpy()
        val_mae = float(np.mean(np.abs(val_pred - val_y)))

        if val_mae < best_val - 1e-5:
            best_val   = val_mae
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            patience   = 0
            used_epochs = epoch + 1
        else:
            patience += 1
            if patience >= MLP_EARLY_STOP:
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    model.eval()
    with torch.no_grad():
        te_X   = tensors["te_X"].to(DEVICE)
        te_sid = tensors["te_sid"].to(DEVICE)
        te_pred_norm = model(te_X, te_sid).cpu().numpy()

    te_pred = te_pred_norm * tensors["te_sd"] + tensors["te_mu"]
    te_pred = np.clip(te_pred, 0.0, None)
    return te_pred, best_val, used_epochs


print(f"Preparing tensors (train: {len(train_part):,} rows, val: {len(val_part):,} rows)...")
tensors = prepare_tensors(train_part, val_part, NUMERIC_FEATS)

print(f"\nFitting MLP-MAE × {len(SEEDS)} seeds on full train...")
t0 = time.time()
mlp_seed_preds = []
for s in SEEDS:
    seed_t = time.time()
    p, best_val_mae, used_epochs = train_mlp_one_seed(tensors, n_numeric=len(NUMERIC_FEATS), seed=s)
    mlp_seed_preds.append(p)
    print(f"  seed {s:>4}: best inner-val L1 = {best_val_mae:.4f}, epochs used = {used_epochs}, "
          f"mean pred = {p.mean():.3f}, time = {time.time() - seed_t:.1f}s")

pred_mlp = np.mean(np.stack(mlp_seed_preds, axis=0), axis=0)
pred_mlp = np.clip(pred_mlp, 0.0, None)
print(f"\nMLP ensemble done in {time.time() - t0:.1f}s.")
print(f"  pred_mlp:  shape {pred_mlp.shape}, range [{pred_mlp.min():.3f}, {pred_mlp.max():.3f}], mean {pred_mlp.mean():.3f}")

Preparing tensors (train: 414,720 rows, val: 32,256 rows)...

Fitting MLP-MAE × 5 seeds on full train...
  seed   42: best inner-val L1 = 0.2726, epochs used = 30, mean pred = 10.642, time = 139.3s
  seed  101: best inner-val L1 = 0.2724, epochs used = 26, mean pred = 10.672, time = 134.1s
  seed  202: best inner-val L1 = 0.2731, epochs used = 30, mean pred = 10.660, time = 134.8s
  seed  303: best inner-val L1 = 0.2732, epochs used = 30, mean pred = 10.622, time = 135.7s
  seed  404: best inner-val L1 = 0.2728, epochs used = 27, mean pred = 10.629, time = 135.2s

MLP ensemble done in 679.1s.
  pred_mlp:  shape (32256,), range [1.167, 40.396], mean 10.645


---
## §5. Apply blend weights, sanity-check, and write submission

Apply the frozen 7b weights, clamp at zero, merge against `forecast_index_validation.csv`, fill any join misses with `series_mean`, and write the two output CSVs:

- `validation_predictions_7c.csv` — three columns (`series_id, timestamp, prediction`) — the leaderboard artifact.
- `val_components_7c.csv` — adds `pred_lgbm, pred_mlp, pred_blend` per row, for any downstream diagnostic work (per-series WAPE study, residual plots, etc.) without re-running the refit.

In [6]:
pred_blend = W_LGBM * pred_lgbm + W_MLP * pred_mlp
pred_blend = np.clip(pred_blend, 0.0, None)

# Sanity check: blend distribution sits between (and similar to) the two components
print("Prediction distributions (mean, std, p50, p95):")
for name, arr in [("LGBM", pred_lgbm), ("MLP", pred_mlp), ("BLEND", pred_blend)]:
    print(f"  {name:>6}: mean = {arr.mean():>6.3f}, std = {arr.std():>6.3f}, "
          f"p50 = {np.median(arr):>6.3f}, p95 = {np.percentile(arr, 95):>6.3f}")

# Cross-check the per-series mean prediction against train per-series mean target.
# Big drift here would mean the model collapsed or the inference frame is misaligned.
train_per_sid = train_df.groupby("series_id")["target"].mean()
val_per_sid_blend = pd.Series(pred_blend, index=val_part["series_id"].values).groupby(level=0).mean()
rel_drift = ((val_per_sid_blend - train_per_sid) / train_per_sid.replace(0, np.nan)).abs()
print(f"\nPer-series |blend_pred_mean − train_target_mean| / train_target_mean:")
print(f"  median = {rel_drift.median():.3f}, p90 = {rel_drift.quantile(0.9):.3f}, max = {rel_drift.max():.3f}")
if rel_drift.median() > 0.5:
    print("  ⚠️  Median per-series drift > 50% — investigate before submitting.")

# Build submission frame and align against the official forecast index.
val_rows = pd.DataFrame({
    "series_id": val_part["series_id"].values,
    "timestamp": val_part["timestamp"].values,
    "prediction": pred_blend,
    "pred_lgbm":  pred_lgbm,
    "pred_mlp":   pred_mlp,
})

submission = val_index.merge(
    val_rows[["series_id", "timestamp", "prediction"]],
    on=["series_id", "timestamp"], how="left",
)
missing = submission["prediction"].isna().sum()
if missing > 0:
    print(f"\n⚠️  {missing} rows have no prediction after the merge — filling with series_mean.")
    sm_map = sm_full.set_index("series_id")["series_mean"]
    submission["prediction"] = submission["prediction"].fillna(submission["series_id"].map(sm_map))

submission["prediction"] = submission["prediction"].clip(lower=0.0)

assert submission["prediction"].isna().sum() == 0, "prediction column still has NaNs after fillna"
assert len(submission) == len(val_index), "submission row count diverged from val_index"

sub_path  = OUTPUT_DIR / "validation_predictions_7c.csv"
comp_path = OUTPUT_DIR / "val_components_7c.csv"
submission[["series_id", "timestamp", "prediction"]].to_csv(sub_path, index=False)
val_rows.assign(w_lgbm=W_LGBM, w_mlp=W_MLP).to_csv(comp_path, index=False)

print(f"\n✅ Wrote {len(submission):,} predictions → {sub_path}")
print(f"✅ Wrote per-model components → {comp_path}")
print("\nFirst 5 submission rows:")
print(submission[["series_id", "timestamp", "prediction"]].head())

Prediction distributions (mean, std, p50, p95):
    LGBM: mean = 10.496, std =  5.157, p50 =  9.714, p95 = 20.118
     MLP: mean = 10.645, std =  5.228, p50 =  9.848, p95 = 20.443
   BLEND: mean = 10.602, std =  5.205, p50 =  9.810, p95 = 20.354

Per-series |blend_pred_mean − train_target_mean| / train_target_mean:
  median = 0.061, p90 = 0.245, max = 0.449

✅ Wrote 32,256 predictions → /kaggle/working/validation_predictions_7c.csv
✅ Wrote per-model components → /kaggle/working/val_components_7c.csv

First 5 submission rows:
  series_id           timestamp  prediction
0  unit_000 2023-06-30 00:00:00    4.351357
1  unit_000 2023-06-30 01:00:00    4.912405
2  unit_000 2023-06-30 02:00:00    5.435150
3  unit_000 2023-06-30 03:00:00    4.662642
4  unit_000 2023-06-30 04:00:00    5.214704


---
## §6. What this submission represents — and what to look for in the leaderboard score

- The frozen weights came from OOF optimization on `oof_predictions_7a_*.csv`. Pooled CV-WAPE of this exact blend on OOF was **14.4683**.
- The historical CV → LB gap in this project has run ~0.1–0.2 WAPE (per `APPROACH.md`). A leaderboard around **14.5–14.7** would be on-trend; anything materially worse than ~14.9 indicates a feature-engineering or refit issue, not a blend-weight issue.
- If the leaderboard comes back at < 14.7, this is the DL-track submission and the blend story is settled. If it comes back at > 14.9, the next investigation is whether the full-train MLP refit drifted (compare `val_components_7c.csv` against a CV-fold MLP prediction on the same val timestamps), *not* whether the blend weights were wrong.
- Either way, the next experimental priority after this submits is the fold-1 per-series WAPE breakdown (open since exp-5 / exp-6) — that's where additional WAPE almost certainly lives, regardless of which model class produces it.